# 05: MILK10k tables and folders

Creates MILK10k's Bronze tables and folders. Run after `00_setup_storage_and_shared_tables`, then upload the source archive to the landing path this notebook prints.

In [ ]:
%run ../_setup_env


In [ ]:
from data_platform.dataset_layout import load_dataset

dataset = load_dataset(CONFIG_ROOT, "milk10k")
DATASET_CONFIG = dataset["config"]
ARCHIVES = dataset["archives"]
landing_paths, bronze_paths, bronze_tables = dataset["landing_paths"], dataset["bronze_paths"], dataset["bronze_tables"]

for archive in ARCHIVES:
    print(f"{archive['source_split']} archive upload path: {archive['archive_dbfs_path']}")
    print(f"{archive['source_split']} Bronze image index table: {bronze_tables['image_index']}")
    print(f"{archive['source_split']} Bronze metadata path: {archive['metadata_target_path']}")


## Create Bronze tables and folders

In [ ]:
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS bronze.milk10k_image_index (
      image_id STRING,
      source_split STRING,
      archive_member_path STRING,
      source_archive_uri STRING,
      byte_length BIGINT,
      source_checksum STRING
    )
    USING DELTA
    """
)
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS bronze.milk10k_source_metadata (
      image_id STRING,
      source_split STRING,
      source_uri STRING,
      attribution STRING,
      copyright_license STRING,
      age_approx STRING,
      anatom_site_1 STRING,
      anatom_site_2 STRING,
      anatom_site_special STRING,
      concomitant_biopsy STRING,
      diagnosis_1 STRING,
      diagnosis_2 STRING,
      diagnosis_3 STRING,
      diagnosis_4 STRING,
      diagnosis_confirm_type STRING,
      image_manipulation STRING,
      image_type STRING,
      lesion_id STRING,
      melanocytic STRING,
      sex STRING,
      source_checksum STRING,
      ingestion_run_id STRING,
      ingested_at TIMESTAMP
    )
    USING DELTA
    """
)
dbutils.fs.mkdirs(landing_paths["dataset_archive"])
for path in bronze_paths.values():
    dbutils.fs.mkdirs(path)
for archive in ARCHIVES:
    dbutils.fs.mkdirs(archive["metadata_dir_path"])

display(spark.sql("SHOW TABLES IN bronze"))
display(dbutils.fs.ls(landing_paths["archives"]))
display(dbutils.fs.ls(bronze_paths["root"]))


## Upload the source archive

Upload each archive listed in `config/bronze/datasets/milk10k.yaml` to the landing path printed above, then run `10_bronze_ingest`.